In [1]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Project modules
import config
from hardware import SLMManager
from phase_generators import generate_fresnel_pattern, generate_optimized_pattern
from visualization import plot_live_update, plot_final_results, plot_fresnel_pattern

# Make sure matplotlib works in notebooks (safe if not in IPython)
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

# --- Global hardware manager ---
slm_manager = SLMManager(sim_mode=True)

# --- UI builder ---
def build_ui():
    # --- Mode selector ---
    mode_selector = widgets.ToggleButtons(
        options=['Optimized Microlens', 'Fresnel Microlens'],
        description='Mode:', style={'description_width': 'initial'}
    )

    # --- Controls for Optimized mode ---
    opt_widgets = {
        name: (widgets.IntText if ('ni' in name or 'rows' in name or 'cols' in name) else widgets.FloatText)(
            value=val, description=desc, style={'description_width': 'initial'}
        )
        for name, (val, desc) in {
            'focal_length_coarse': (config.UI_DEFAULTS['focal_length_coarse'], 'Focal Length (mm)'),
            'focal_length_fine':   (config.UI_DEFAULTS['focal_length_fine'],   'Fine Adjust (mm)'),
            'rows':                (config.UI_DEFAULTS['rows'],                'Rows'),
            'cols':                (config.UI_DEFAULTS['cols'],                'Cols'),
            'overlap_ratio':       (config.UI_DEFAULTS['overlap_ratio'],       'Overlap Ratio'),
            'dof_factor':          (config.UI_DEFAULTS['dof_factor'],          'Depth-of-Field Factor'),
            'size_factor':         (config.UI_DEFAULTS['size_factor'],         'Size Factor'),
            'psf_energy_level':    (config.UI_DEFAULTS['psf_energy_level'],    'PSF Energy'),
            'phase_range':         (config.UI_DEFAULTS['phase_range'],         'Phase Range'),
            'lr':                  (config.UI_DEFAULTS['lr'],                  'Learning Rate'),
            'ni':                  (config.UI_DEFAULTS['ni'],                  'Iterations'),
        }.items()
    }
    opt_widgets['lens_type'] = widgets.ToggleButtons(
        options=[('Convex (+)', False), ('Concave (-)', True)],
        description='Lens Type:', style={'description_width': 'initial'}
    )
    optimizer_controls = widgets.VBox(list(opt_widgets.values()), layout={'display': 'flex'})

    # --- Controls for Fresnel mode ---
    fresnel_widgets = {
        name: (widgets.IntText if 'two_pi_value' in name else widgets.FloatText)(
            value=val, description=desc, style={'description_width': 'initial'}
        )
        for name, (val, desc) in {
            'focal_length_coarse': (config.FRESNEL_DEFAULTS['focal_length_coarse'], 'Focal Length (mm)'),
            'rows':                (config.FRESNEL_DEFAULTS['rows'],                'Rows'),
            'cols':                (config.FRESNEL_DEFAULTS['cols'],                'Cols'),
            'roi_width':           (config.FRESNEL_DEFAULTS['roi_width'],           'ROI Width'),
            'roi_height':          (config.FRESNEL_DEFAULTS['roi_height'],          'ROI Height'),
            'angle_x_mrad':        (config.FRESNEL_DEFAULTS['angle_x_mrad'],        'Deflection X (mrad)'),
            'angle_y_mrad':        (config.FRESNEL_DEFAULTS['angle_y_mrad'],        'Deflection Y (mrad)'),
            'two_pi_value':        (config.FRESNEL_DEFAULTS['two_pi_value'],        'Gray Value for 2π'),
        }.items()
    }
    fresnel_widgets['lens_type'] = widgets.ToggleButtons(
        options=[('Convex (+)', False), ('Concave (-)', True)],
        description='Lens Type:', style={'description_width': 'initial'}
    )
    fresnel_controls = widgets.VBox(list(fresnel_widgets.values()), layout={'display': 'none'})

    # --- Dynamic UI toggle ---
    def on_mode_change(change):
        if change.new == 'Optimized Microlens':
            optimizer_controls.layout.display = 'flex'
            fresnel_controls.layout.display = 'none'
        else:
            optimizer_controls.layout.display = 'none'
            fresnel_controls.layout.display = 'flex'
    mode_selector.observe(on_mode_change, names='value')

    # --- Main execution ---
    output_widget = widgets.Output()

    def run_simulation(_b):
        with output_widget:
            clear_output(wait=True)
            mode = mode_selector.value
            print(f"Running mode: {mode}")

            if mode == 'Optimized Microlens':
                params = {key: w.value for key, w in opt_widgets.items()}
                params['shape'] = slm_manager.shape  # square ROI
                phi, optimizer_obj = generate_optimized_pattern(params, plot_live_update)
                slm_manager.upload(phi)
                plot_final_results(optimizer_obj, optimizer_obj.phase_param.detach())

            elif mode == 'Fresnel Microlens':
                params = {key: w.value for key, w in fresnel_widgets.items()}
                params['shape'] = slm_manager.shape
                # Combine focal length sign with lens type
                focal_len = params['focal_length_coarse']
                params['focal_length'] = -focal_len if params['lens_type'] else focal_len
                # Fill in ROI center defaults
                params['roi_center_x'] = slm_manager.shape[1] // 2
                params['roi_center_y'] = slm_manager.shape[0] // 2

                phi, info_dict = generate_fresnel_pattern(params)
                slm_manager.upload(phi)
                plot_fresnel_pattern(info_dict)

    apply_button = widgets.Button(
        description='Generate & Upload',
        button_style='primary',
        icon='cogs'
    )
    apply_button.on_click(run_simulation)

    # --- Final layout ---
    ui = widgets.VBox([
        mode_selector,
        widgets.HTML("<hr>"),
        optimizer_controls,
        fresnel_controls,
        widgets.HTML("<hr>"),
        apply_button,
        output_widget
    ])
    display(ui)

# --- Run UI ---
build_ui()
